# 5.6. Dropout
D2L의 Dropout장을 PyTorch 기준으로 정리함.


## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Dropout은 왜 필요한가?

신경망을 학습할 때 우리가 원하는 것은 단순히 train data를 맞추는 게 아니라 새로운 데이터에서 잘 작동하는 것을 원한다.

하지만 모델이 너무 복잡하면 학습 데이터에 지나치게 맞춰질 수 있다. (Overfitting)

앞에서 Weight Decay를 배웠는데 이번엔 다른 방법인 Dropout을 알아보자.

Dropout의 아이디어는 단순하다. 학습하는 동안 일부 뉴런의 출력을 랜덤하게 0으로 만든다.

예를 들어서 은닉층에 뉴런이 5개 있으면

```text
h1 h2 h3 h4 h5
|  |  |  |  |
사용 사용 사용 사용 사용

Dropout
h1 h2 h3 h4 h5
|     |     |
사용 제거 사용 제거 사용

일부 뉴런이 랜덤하게 제거된다.
```

뉴런 자체를 삭제하는게 아니고 해당 학습 단계에서 출력값을 0으로 만든다.

D2L에서는 좋은 모델이 작은 입력 변화에도 지나치게 민감하지 않아야 한다는 관점에서 Dropout을 소개하고, 학습 중 일부 hidden unit을 무작위로 0으로 만들어 노이즈를 주입하는 정규화 기법으로 설명한다.

## 2. 왜 뉴런을 일부로 끄나?

예를 들어서 어떤 신경망이 있다고 했을때

```text
입력
 ↓
[뉴런 A, 뉴런 B, 뉴런 C]
 ↓
출력
```

학습 과정에서 모델이 뉴런 A만 보고 정답을 맞힐 수 있다고 학습을 해버릴 수도 있다. 그러면 모델이 A에 지나치게 의존하게 되는데 Dropout을 적용하면 학습할 때마다 뉴런이 계속 달라진다.

```text
1번째 학습
A O
B X
C O

2번째 학습
A X
B O
C O

3번째 학습
A O
B O
C X
```

그래서 하나에만 의존하지 않는 걸 학습하게 된다. 여러 뉴런이 함께 유용한 특징을 학습하도록 유도할 수 있다. 

D2L에서는 hidden activation 조합에 지나치게 의존하는 현상을 `co-adaptation`이라고 설명하고, Dropout이 이런 의존성을 끊는 직관을 제공한다고 설명한다.

## 3. Dropout 확률 p

Dropout에는 중요한 값 p가 있다. p는 뉴런을 제거할 확률을 의미한다.

    nn.Dropout(p=0.5)

라면 뉴런 출력이 50%확률로 제거된다. 정확히 절반을 제거하는게 아니라 각 값마다 50% 확률로 제거할지 결정한다는 뜻이다. 그래서 매번 제거되는 뉴런은 달라질 수 있다.

## 4. 살아남은 값이 커지는 이유

Dropout은 단순히 값을 0으로 만드는 것으로 끝나지 않는다.

Dropout 확률을 `p`라고 하면 살아남는 값은

```text
1 / (1 - p) 배 한다. (Dropout때문에 전체 출력 크기가 평균적으로 작아지는것을 방지)
```

예를 들어서

p - 0.5

면 살아남을 확률은

1 - p = 0.5

이고 살아남는 값에 

1 / 0.5 = 2배를 해준다.

예를 들어서 원래 activation이 아래이고
    [2, 4, 6, 8]

Dropout과정에서 일부가 제거되면
    [2, 0, 6, 0]

이렇게 될 수 있다. 살아남은 값 2배를 하면 이렇게 된다.
    [4, 0, 12, 0]

D2L에선 Dropout을 이렇게 정의한다. Dropout 확률이 $p$일 때 activation $h$는 확률 $p$로 0이 되고, 살아남으면 $h/(1-p)$가 된다. 이렇게 하면 평균적인 값이 원래 activation과 같아진다.

$$
h' =
\begin{cases}0 & \text{확률 } p \\
\frac{h}{1-p} & \text{확률 } 1-p
\end{cases}
$$

## 5. Dropout 구현해보기

D2L에서도 Dropout을 구현해 원리를 확인한다.

In [3]:
def dropout_layer(X, p):
    assert 0 <= p <= 1

    if p == 1:
        return torch.zeros_like(X)

    if p == 0:
        return X

    # p보다 크면 살아남음
    mask = (torch.rand_like(X) > p).float() # rand_like(X)는 X와 같은 랜덤 숫자를 생성한다.

    # 살아남은 값은 1 / (1-p) 배
    return X * mask / (1 - p)

예를 들어서
X = [1, 2, 3, 4]
torch.rand_like(X) = [0.8, 0.2, 0.7, 0.1]

p = 0.5면 torch.rand_like(X) > 0.5를 계산하고 결과는 [True, False, True, False]이고 이건 [1, 0, 1, 0]이라는 mask이다.

X * mask를 하면 [1, 0, 3, 0]이 된다.

마지막으로 살아남은 값을 1-p로 나누어 크기를 보정해준다.

In [11]:
X = torch.arange(1, 9, dtype=torch.float32)

print("원본")
print(X)

print("\np=0")
print(dropout_layer(X, 0))

print("\np=0.5")
print(dropout_layer(X, 0.5))

print("\np=1")
print(dropout_layer(X, 1))

원본
tensor([1., 2., 3., 4., 5., 6., 7., 8.])

p=0
tensor([1., 2., 3., 4., 5., 6., 7., 8.])

p=0.5
tensor([ 2.,  0.,  6.,  0., 10.,  0., 14.,  0.])

p=1
tensor([0., 0., 0., 0., 0., 0., 0., 0.])


## 6. MLP에서 Dropout은 어디에 넣는가?

기존 MLP 흐름

```text
입력
 ↓
Linear
 ↓
ReLU  
 ↓
       <- 보통 여기에 넣는다고 한다.
Linear
 ↓
ReLU
 ↓
Linear
 ↓
출력
```

    Linear -> ReLU -> Dropout

Dropout은 뉴런의 activation을 랜덤하게 제거하기 때문에 일반적으로 hidden layer의 activation 뒤에 적용한다.

D2L 예제에서도 각 hidden layer에서 `Linear -> ReLU -> Dropout`형태로 적용한다.

## 7. PyTorch에서 Dropout 쓰기

PyTorch는 `nn.Dropout`을 쓰면 된다.

In [12]:
model = nn.Sequential(
    nn.Flatten(),

    nn.Linear(28 * 28, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5),

    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5),

    nn.Linear(256, 10)
)

model

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): Dropout(p=0.5, inplace=False)
  (7): Linear(in_features=256, out_features=10, bias=True)
)

Shape 흐름

```text
입력 이미지
[batch_size, 1, 28, 28]

        ↓ Flatten

[batch_size, 784]

        ↓ Linear(784, 256)

[batch_size, 256]

        ↓ ReLU

[batch_size, 256]

        ↓ Dropout

[batch_size, 256]

        ↓ Linear(256, 256)

[batch_size, 256]

        ↓ ReLU

[batch_size, 256]

        ↓ Dropout

[batch_size, 256]

        ↓ Linear(256, 10)

[batch_size, 10]
```

Dropout은 shape를 바꾸지 않는다.

## 8. 학습할 때만 Dropout 사용

Dropout은 학습할 때만 사용한다. Dropout은 학습 과정에서 모델을 일부로 불안정하게 만들어 특정 뉴런에 의존하지 못하도록 하는 것이기 때문이다.

학습이 끝나고 실제 예측할 때까지 뉴런을 랜덤하게 제거할 필요 없다.

D2L에서도 일반적인 Dropout에서는 테스트 시에 뉴런을 제거하지 않는다고 설명한다.

In [14]:
x = torch.ones(1, 10)

dropout = nn.Dropout(p=0.5)

# 학습 모드
dropout.train()

print("Training")
print(dropout(x))

Training
tensor([[0., 2., 0., 0., 2., 2., 2., 0., 0., 0.]])


In [15]:
dropout.eval()

print("Evaluation")
print(dropout(x))

Evaluation
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])


## 9. 과적합 방지 기법 정리

### Weight Decay

가중치가 지나치게 커지는 것을 억제한다. (Weight 자체 제어)

```text
큰 Weight에 penalty
↓
Weight를 작게 유지
↓
모델이 지나치게 복잡해지는 것을 억제
```

### Dropout

학습할 때 일부 뉴런의 출력을 랜덤하게 제거한다. (Activation을 랜덤 제거)

```text
일부 뉴런 제거
↓
특정 뉴런에 의존하기 어려움
↓
보다 분산된 특징 학습
```

둘다 `Overfitting`을 줄이고 `Generalization`을 높이기 위한 `Regularization` 방법이다.

## 10. 오늘의 정리

- Dropout은 신경망의 Overfitting을 줄이기 위한 Regularization 기법이다.
- 학습 중 일부 뉴런의 activation을 랜덤하게 0으로 만든다.
- `p`는 뉴런을 제거할 확률이다.
- `p=0.5`라면 각 activation이 50% 확률로 제거된다.
- 살아남은 activation은 `1 / (1-p)`배 하여 평균적인 크기를 유지한다.
- Dropout은 tensor의 shape을 변경하지 않고 일부 값만 0으로 만든다.
- MLP에서는 보통 `Linear → ReLU → Dropout` 순서로 사용한다.
- PyTorch에서는 `nn.Dropout(p)`으로 사용할 수 있다.
- Dropout은 `model.train()`일 때 활성화되고 `model.eval()`일 때 비활성화된다.
- Weight Decay는 Weight를 제어하고, Dropout은 Activation을 랜덤하게 제거한다
- 둘 모두 학습 데이터를 외우는 것을 줄이고 새로운 데이터에 대한 Generalization을 높이는 것이 목적이다.